<a href="https://colab.research.google.com/github/upgrade-projects/pysparks-projects/blob/main/Enterprise_GenAI_Adoption_Impact.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Enterprise GenAI Adoption & Workforce Impact Data**

You are provided with a dataset titled ‘Enterprise GenAI Adoption & Workforce Impact Data’, which captures detailed information about enterprise-level adoption of Generative AI tools across industries and countries. This dataset includes company demographics, GenAI tools adopted, year of adoption, workforce training efforts, changes in productivity and employee sentiment regarding AI integration.

 Dataset [Link](https://drive.google.com/file/d/1WLcC9oYgGWBEBlM9Yx_fk3snHl4pw8ia/view?usp=drive_link)

In [ ]:
!pip install pyspark
!pip install findspark
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower, regexp_replace, sum, count, avg, countDistinct, array_contains, trim, length, substring, concat_ws, lit
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover
from pyspark.ml.clustering import LDA
from pyspark.ml.feature import CountVectorizer
from pyspark.sql import functions as F

# Create a SparkSession
spark = SparkSession.builder.appName("GenAI_Adoption_Analysis").getOrCreate()

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load the dataset
data_path = '/content/drive/MyDrive/Colab_Notebooks/Upgrad_Training/data/Enterprise_GenAI_Adoption_Impact.csv'
df = spark.read.csv(data_path, header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- Company Name: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- GenAI Tool: string (nullable = true)
 |-- Adoption Year: integer (nullable = true)
 |-- Number of Employees Impacted: integer (nullable = true)
 |-- New Roles Created: integer (nullable = true)
 |-- Training Hours Provided: integer (nullable = true)
 |-- Productivity Change (%): double (nullable = true)
 |-- Employee Sentiment: string (nullable = true)

+--------------------+-----------+------------+----------+-------------+----------------------------+-----------------+-----------------------+-----------------------+--------------------+
|        Company Name|   Industry|     Country|GenAI Tool|Adoption Year|Number of Employees Impacted|New Roles Created|Training Hours Provided|Productivity Change (%)|  Employee Sentiment|
+--------------------+-----------+------------+----------+-------------+----------------------------+-----------------+--------------

With the given dataset, solve the following tasks:


**Task 1: Write a function that returns the number of rows, the number of columns, the list of unique GenAI tools used and the number of distinct industries in the dataset.**

In [ ]:
def get_dataset_summary(df):
    """
    Returns a summary of the dataset including number of rows, columns,
    unique GenAI tools, and distinct industries.
    """
    num_rows = df.count()
    num_columns = len(df.columns)

    # Use the correct column name 'GenAI Tool' as inferred from the schema
    unique_genai_tools = df.select('GenAI Tool').distinct().rdd.flatMap(lambda x: x).collect()
    num_distinct_industries = df.select('Industry').distinct().count()

    return {
        "Number of Rows": num_rows,
        "Number of Columns": num_columns,
        "Unique GenAI Tools": unique_genai_tools,
        "Number of Distinct Industries": num_distinct_industries
    }

# Call the function and print the summary
dataset_summary = get_dataset_summary(df)
print(dataset_summary)

{'Number of Rows': 100000, 'Number of Columns': 10, 'Unique GenAI Tools': ['ChatGPT', 'Gemini', 'LLaMA', 'Claude', 'Groq', 'Mixtral'], 'Number of Distinct Industries': 14}


**Task 2: Write a function that standardises column names by converting them to lowercase and replacing spaces with underscores.**

In [ ]:
def standardize_column_names(df):
    """
    Standardizes column names by converting them to lowercase
    and replacing spaces with underscores.
    """
    original_columns = df.columns
    new_columns = [col.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('%', '') for col in original_columns]

    # Create a mapping for renaming
    mapping = dict(zip(original_columns, new_columns))

    # Rename columns
    for old_name, new_name in mapping.items():
        df = df.withColumnRenamed(old_name, new_name)

    return df

# Apply the function
df_standardized = standardize_column_names(df)
print("Original Columns:", df.columns)
print("Standardized Columns:", df_standardized.columns)
df_standardized.printSchema()

Original Columns: ['Company Name', 'Industry', 'Country', 'GenAI Tool', 'Adoption Year', 'Number of Employees Impacted', 'New Roles Created', 'Training Hours Provided', 'Productivity Change (%)', 'Employee Sentiment']
Standardized Columns: ['company_name', 'industry', 'country', 'genai_tool', 'adoption_year', 'number_of_employees_impacted', 'new_roles_created', 'training_hours_provided', 'productivity_change_', 'employee_sentiment']
root
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- country: string (nullable = true)
 |-- genai_tool: string (nullable = true)
 |-- adoption_year: integer (nullable = true)
 |-- number_of_employees_impacted: integer (nullable = true)
 |-- new_roles_created: integer (nullable = true)
 |-- training_hours_provided: integer (nullable = true)
 |-- productivity_change_: double (nullable = true)
 |-- employee_sentiment: string (nullable = true)



**Task 3: Write a function that returns a dictionary with column names as keys and a count of null or missing values as values.**

In [ ]:
def count_null_values(df):
    """
    Returns a dictionary with column names as keys and a count of null or missing values as values.
    """
    null_counts = {}
    for col_name in df.columns:
        # Counting nulls: PySpark's isNull() checks for actual null values.
        # For empty strings or whitespace that might represent missing data,
        # additional checks would be needed if that's considered 'missing'.
        # For now, we focus on strictly null values.
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            null_counts[col_name] = null_count
    return null_counts

# Apply the function to the standardized DataFrame
missing_values_summary = count_null_values(df_standardized)
print("Missing Values Summary:", missing_values_summary)

Missing Values Summary: {}


**Task 4: Write a function that casts adoption_year to IntegerType, productivity_change to FloatType and training_hours_provided and number_of_employees_impacted to IntegerType.**

In [ ]:
def cast_column_types(df):
    """
    Casts specified columns to their appropriate data types.
    - adoption_year to IntegerType
    - productivity_change_ to FloatType
    - training_hours_provided to IntegerType
    - number_of_employees_impacted to IntegerType
    """
    df = df.withColumn("adoption_year", col("adoption_year").cast(IntegerType()))
    # Note: 'productivity_change_' is the standardized name for 'Productivity Change (%)'
    df = df.withColumn("productivity_change_", col("productivity_change_").cast(FloatType()))
    df = df.withColumn("training_hours_provided", col("training_hours_provided").cast(IntegerType()))
    df = df.withColumn("number_of_employees_impacted", col("number_of_employees_impacted").cast(IntegerType()))
    return df

# Apply the function
df_casted = cast_column_types(df_standardized)
print("Schema after type casting:")
df_casted.printSchema()

Schema after type casting:
root
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- country: string (nullable = true)
 |-- genai_tool: string (nullable = true)
 |-- adoption_year: integer (nullable = true)
 |-- number_of_employees_impacted: integer (nullable = true)
 |-- new_roles_created: integer (nullable = true)
 |-- training_hours_provided: integer (nullable = true)
 |-- productivity_change_: float (nullable = true)
 |-- employee_sentiment: string (nullable = true)



**Task 5: Write a function that adds a new column adoption_level based on number_of_employees_impacted: High if greater than 5000, medium if between 1000 and  5000 and low if less than 1000.**

In [ ]:
def add_adoption_level(df):
    """
    Adds a new column 'adoption_level' based on 'number_of_employees_impacted':
    - High if greater than 5000
    - Medium if between 1000 and 5000 (inclusive of 1000, exclusive of 5000)
    - Low if less than 1000
    """
    df = df.withColumn("adoption_level",
                      when(col("number_of_employees_impacted") > 5000, "High")
                      .when((col("number_of_employees_impacted") >= 1000) & (col("number_of_employees_impacted") <= 5000), "Medium")
                      .otherwise("Low"))
    return df

# Apply the function
df_with_adoption_level = add_adoption_level(df_casted)
print("Schema with new 'adoption_level' column:")
df_with_adoption_level.printSchema()
df_with_adoption_level.select("number_of_employees_impacted", "adoption_level").show(5)

Schema with new 'adoption_level' column:
root
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- country: string (nullable = true)
 |-- genai_tool: string (nullable = true)
 |-- adoption_year: integer (nullable = true)
 |-- number_of_employees_impacted: integer (nullable = true)
 |-- new_roles_created: integer (nullable = true)
 |-- training_hours_provided: integer (nullable = true)
 |-- productivity_change_: float (nullable = true)
 |-- employee_sentiment: string (nullable = true)
 |-- adoption_level: string (nullable = false)

+----------------------------+--------------+
|number_of_employees_impacted|adoption_level|
+----------------------------+--------------+
|                        5277|          High|
|                       18762|          High|
|                       11307|          High|
|                       18834|          High|
|                        8364|          High|
+----------------------------+--------------+
only showing 

**Task 6: Write a function that groups the dataset by country and industry, and returns the total number of companies, average productivity change and total new roles created.**

In [ ]:
def get_grouped_summary(df):
    """
    Groups the dataset by country and industry, and returns the total number of companies,
    average productivity change and total new roles created.
    """
    grouped_df = df.groupBy("country", "industry").agg(
        countDistinct("company_name").alias("total_companies"),
        avg("productivity_change_").alias("average_productivity_change"),
        sum("new_roles_created").alias("total_new_roles_created")
    )
    return grouped_df

# Apply the function
df_grouped_summary = get_grouped_summary(df_with_adoption_level)
print("Grouped Summary by Country and Industry:")
df_grouped_summary.show(5)

Grouped Summary by Country and Industry:
+-------+-----------+---------------+---------------------------+-----------------------+
|country|   industry|total_companies|average_productivity_change|total_new_roles_created|
+-------+-----------+---------------+---------------------------+-----------------------+
|    USA|  Education|            510|          18.64901960363575|                   8186|
|     UK|  Education|            504|         17.515079372458988|                   7709|
| Brazil|  Education|            537|         18.544320270335874|                   8216|
| Brazil| Healthcare|            553|          18.20415912010786|                   8687|
|    USA|Hospitality|            488|         18.550409882283603|                   7483|
+-------+-----------+---------------+---------------------------+-----------------------+
only showing top 5 rows


**Task 7: Write a function that preprocesses the employee_sentiment column by converting it to lowercase and removing punctuation.**

In [ ]:
def preprocess_employee_sentiment(df):
    """
    Preprocesses the employee_sentiment column by converting it to lowercase
    and removing punctuation.
    """
    # Convert to lowercase
    df = df.withColumn("employee_sentiment", lower(col("employee_sentiment")))
    # Remove punctuation using a regular expression
    df = df.withColumn("employee_sentiment", regexp_replace(col("employee_sentiment"), "[\\p{Punct}]", ""))
    return df

# Apply the function
df_processed_sentiment = preprocess_employee_sentiment(df_with_adoption_level)
print("Employee Sentiment after preprocessing (first 5 rows):")
df_processed_sentiment.select("employee_sentiment").show(5, truncate=False)

Employee Sentiment after preprocessing (first 5 rows):
+------------------------------------------------------------------------------------------------------------------+
|employee_sentiment                                                                                                |
+------------------------------------------------------------------------------------------------------------------+
|productivity increased but theres anxiety about job security job rotations have increased                         |
|we now finish tasks faster but some older employees are struggling clients now expect faster results due to ai use|
|productivity increased but theres anxiety about job security it changed how we communicate internally             |
|ai helped me reduce repetitive tasks but learning curve was steep it changed how we communicate internally        |
|job roles have shifted a lot which is both good and scary clients now expect faster results due to ai use         |
+--------

**Task 8: Write a function that returns a year-wise summary including number of companies, average training hours and the most adopted GenAI tool for each year.**

In [ ]:
from pyspark.sql.window import Window

def get_year_wise_summary(df):
    """
    Returns a year-wise summary including number of companies,
    average training hours and the most adopted GenAI tool for each year.
    """
    # 1. Aggregate for number of companies and average training hours
    yearly_summary_base = df.groupBy("adoption_year").agg(
        F.countDistinct("company_name").alias("total_companies"),
        F.avg("training_hours_provided").alias("average_training_hours")
    )

    # 2. Determine the most adopted GenAI tool per year
    # Count occurrences of each GenAI tool per year
    genai_tool_counts = df.groupBy("adoption_year", "genai_tool").count()

    # Define a window specification to rank tools within each year.
    # If there's a tie in count, it will pick one arbitrarily (based on `genai_tool` name due to second orderby).
    window_spec = Window.partitionBy("adoption_year").orderBy(F.col("count").desc(), F.col("genai_tool"))

    # Apply the window function to get the top tool (rank 1) for each year
    most_adopted_tool_per_year = genai_tool_counts \
        .withColumn("rank", F.rank().over(window_spec)) \
        .filter(F.col("rank") == 1) \
        .select("adoption_year", F.col("genai_tool").alias("most_adopted_genai_tool"))

    # 3. Join the base summary with the most adopted tool
    final_summary = yearly_summary_base.join(most_adopted_tool_per_year, "adoption_year", "left")

    return final_summary

# Apply the function
df_year_wise_summary = get_year_wise_summary(df_processed_sentiment)
print("Year-wise Summary:")
df_year_wise_summary.orderBy("adoption_year").show()

Year-wise Summary:
+-------------+---------------+----------------------+-----------------------+
|adoption_year|total_companies|average_training_hours|most_adopted_genai_tool|
+-------------+---------------+----------------------+-----------------------+
|         2022|          33180|    12797.869469559975|                   Groq|
|         2023|          33344|    12717.287817898272|                   Groq|
|         2024|          33476|    12712.635709164775|                 Gemini|
+-------------+---------------+----------------------+-----------------------+



**Task 9: Write a function that returns a cleaned version of the dataset with standardised column names, missing values handled, adoption_level column added and employee_sentiment trimmed to 100 characters.**

In [ ]:
def clean_dataset(df):
    """
    Returns a cleaned version of the dataset with:
    1. Standardised column names.
    2. Missing values handled (already none in this dataset).
    3. 'adoption_level' column added.
    4. 'employee_sentiment' trimmed to 100 characters.
    """
    # 1. Standardize column names
    df_cleaned = standardize_column_names(df)

    # 2. Cast column types
    df_cleaned = cast_column_types(df_cleaned)

    # 3. Add adoption_level column
    df_cleaned = add_adoption_level(df_cleaned)

    # 4. Preprocess employee_sentiment (lowercase and remove punctuation)
    df_cleaned = preprocess_employee_sentiment(df_cleaned)

    # 5. Trim employee_sentiment to 100 characters
    df_cleaned = df_cleaned.withColumn("employee_sentiment",
                                       substring(col("employee_sentiment"), 1, 100))

    return df_cleaned

# Apply the function to the original DataFrame
df_final_cleaned = clean_dataset(df)

print("Schema of the final cleaned DataFrame:")
df_final_cleaned.printSchema()
print("First 5 rows of the final cleaned DataFrame with trimmed employee_sentiment:")
df_final_cleaned.select("company_name", "employee_sentiment").show(5, truncate=False)

Schema of the final cleaned DataFrame:
root
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- country: string (nullable = true)
 |-- genai_tool: string (nullable = true)
 |-- adoption_year: integer (nullable = true)
 |-- number_of_employees_impacted: integer (nullable = true)
 |-- new_roles_created: integer (nullable = true)
 |-- training_hours_provided: integer (nullable = true)
 |-- productivity_change_: float (nullable = true)
 |-- employee_sentiment: string (nullable = true)
 |-- adoption_level: string (nullable = false)

First 5 rows of the final cleaned DataFrame with trimmed employee_sentiment:
+---------------------------------------+----------------------------------------------------------------------------------------------------+
|company_name                           |employee_sentiment                                                                                  |
+---------------------------------------+-------------------------

**Task 10: Write a function that returns a star schema design with one fact table and three dimension tables: Company, time and genai_tool.**

In [ ]:
def create_star_schema(df):
    """
    Creates a star schema design with one fact table and three dimension tables:
    Company, Time, and GenAI Tool.
    """
    # 1. Company Dimension Table
    company_dim = df.select("company_name", "industry", "country") \
                      .distinct() \
                      .withColumn("company_id", F.monotonically_increasing_id())

    # 2. Time Dimension Table
    time_dim = df.select("adoption_year") \
                   .distinct() \
                   .withColumn("time_id", F.monotonically_increasing_id())

    # 3. GenAI Tool Dimension Table
    genai_tool_dim = df.select("genai_tool") \
                         .distinct() \
                         .withColumn("genai_tool_id", F.monotonically_increasing_id())

    # 4. Fact Table
    # Join df with dimension tables to get their IDs
    fact_table = df.join(company_dim, ["company_name", "industry", "country"], "left") \
                     .join(time_dim, ["adoption_year"], "left") \
                     .join(genai_tool_dim, ["genai_tool"], "left") \
                     .select(
                         "company_id",
                         "time_id",
                         "genai_tool_id",
                         "number_of_employees_impacted",
                         "new_roles_created",
                         "training_hours_provided",
                         "productivity_change_",
                         "employee_sentiment",
                         "adoption_level"
                     )

    return company_dim, time_dim, genai_tool_dim, fact_table

# Apply the function
company_dim_df, time_dim_df, genai_tool_dim_df, fact_table_df = create_star_schema(df_final_cleaned)

print("Company Dimension Table Schema:")
company_dim_df.printSchema()
print("Company Dimension Table (first 5 rows):")
company_dim_df.show(5)

print("Time Dimension Table Schema:")
time_dim_df.printSchema()
print("Time Dimension Table (first 5 rows):")
time_dim_df.show(5)

print("GenAI Tool Dimension Table Schema:")
genai_tool_dim_df.printSchema()
print("GenAI Tool Dimension Table (first 5 rows):")
genai_tool_dim_df.show(5)

print("Fact Table Schema:")
fact_table_df.printSchema()
print("Fact Table (first 5 rows):")
fact_table_df.show(5)

Company Dimension Table Schema:
root
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- country: string (nullable = true)
 |-- company_id: long (nullable = false)

Company Dimension Table (first 5 rows):
+--------------------+-----------+------------+----------+
|        company_name|   industry|     country|company_id|
+--------------------+-----------+------------+----------+
|Nichols Group Pvt...|     Retail|       Japan|         0|
|Robbins, Henry an...| Healthcare|       India|         1|
| Patel Inc Pvt. Ltd.|    Defense|South Africa|         2|
|Chang, Bradley an...|  Education|         UAE|         3|
|Walker, Cox and B...|Advertising| Switzerland|         4|
+--------------------+-----------+------------+----------+
only showing top 5 rows
Time Dimension Table Schema:
root
 |-- adoption_year: integer (nullable = true)
 |-- time_id: long (nullable = false)

Time Dimension Table (first 5 rows):
+-------------+-------+
|adoption_year|time_id|